In [1]:
import tkinter as tk
from tkinter import messagebox
import requests

# ---------------- CONFIG ----------------
API_KEY = "a67bc225a0377cf1e42ebddf322f6dc1"

WEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"
FORECAST_URL = "https://api.openweathermap.org/data/2.5/forecast"


# ---------------- CURRENT WEATHER ----------------
def get_weather():
    city = city_entry.get().strip()

    if not city:
        messagebox.showerror("Error", "Please enter a city name")
        return

    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    try:
        response = requests.get(WEATHER_URL, params=params, timeout=5)
        data = response.json()

        if response.status_code != 200:
            messagebox.showerror("Error", data.get("message", "City not found"))
            return

        city_name = data["name"]
        temp = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        weather = data["weather"][0]["description"].title()
        wind = data["wind"]["speed"]

        result_label.config(
            text=f"""
📍 City: {city_name}
🌡 Temperature: {temp} °C
🌥 Weather: {weather}
💧 Humidity: {humidity} %
🌬 Wind Speed: {wind} m/s
"""
        )

    except Exception as e:
        messagebox.showerror("Error", str(e))


# ---------------- 5-DAY FORECAST ----------------
def get_forecast():
    city = city_entry.get().strip()

    if not city:
        messagebox.showerror("Error", "Please enter a city name")
        return

    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    try:
        response = requests.get(FORECAST_URL, params=params, timeout=5)
        data = response.json()

        if response.status_code != 200:
            messagebox.showerror("Error", data.get("message", "City not found"))
            return

        forecast_text = f"📍 5-Day Forecast for {city.title()}\n\n"

        # every 8th item = 24 hours
        for i in range(0, 40, 8):
            item = data["list"][i]
            date = item["dt_txt"].split(" ")[0]
            temp = item["main"]["temp"]
            desc = item["weather"][0]["description"].title()

            forecast_text += f"{date}: {temp}°C, {desc}\n"

        result_label.config(text=forecast_text)

    except Exception as e:
        messagebox.showerror("Error", str(e))


# ---------------- MULTI CITY COMPARISON ----------------
def compare_weather():
    cities = city_entry.get().strip()

    if not cities:
        messagebox.showerror("Error", "Enter cities separated by commas")
        return

    city_list = [c.strip() for c in cities.split(",")]
    result_text = ""

    for city in city_list:
        params = {
            "q": city,
            "appid": API_KEY,
            "units": "metric"
        }

        try:
            response = requests.get(WEATHER_URL, params=params, timeout=5)
            data = response.json()

            if response.status_code != 200:
                result_text += f"\n❌ {city}: Not found\n"
                continue

            city_name = data["name"]
            temp = data["main"]["temp"]
            humidity = data["main"]["humidity"]
            weather = data["weather"][0]["description"].title()

            result_text += f"""
📍 {city_name}
🌡 Temp: {temp}°C
🌥 {weather}
💧 Humidity: {humidity}%

----------------------
"""

        except:
            result_text += f"\n⚠ {city}: Error\n"

    result_label.config(text=result_text)


# ---------------- GUI ----------------
window = tk.Tk()
window.title("Weather App")
window.geometry("600x600")
window.resizable(False, False)

title = tk.Label(window, text="🌦 Weather App", font=("Arial", 18, "bold"))
title.pack(pady=10)

hint = tk.Label(window, text="Enter city OR multiple cities (comma separated)", font=("Arial", 10))
hint.pack()

city_entry = tk.Entry(window, font=("Arial", 14), justify="center", width=30)
city_entry.pack(pady=10)

# Buttons
search_btn = tk.Button(window, text="Get Weather", font=("Arial", 12), command=get_weather)
search_btn.pack(pady=5)

forecast_btn = tk.Button(window, text="Get 5-Day Forecast", font=("Arial", 12), command=get_forecast)
forecast_btn.pack(pady=5)

compare_btn = tk.Button(window, text="Compare Cities", font=("Arial", 12), command=compare_weather)
compare_btn.pack(pady=5)

# Output
result_label = tk.Label(window, text="", font=("Arial", 11), justify="left")
result_label.pack(pady=20)

window.mainloop()


: 